# 🚀 SLEEP2K AI CLONE WORKER (48kHz ULTRA STUDIO)

Bộ máy AI Sao Chép Giọng Siêu Thực 98% · Chạy 1-Click Trên Google Colab GPU T4

**Bạn chỉ cần bấm 1 nút Play (▶) ở code bên dưới là XONG!** Hệ thống sẽ tự động cài đặt, tự động kết nối thông về website: https://sleep2k-1.onrender.com 100% tự động


In [ ]:
# ✅ SLEEP2K AI CLONE WORKER - 1-CLICK SETUP
# Tự động cài đặt & tự động kết nối với website https://sleep2k-1.onrender.com
import subprocess, sys, os, time, re

# ── BƯỚC 0: Fix onnxruntime TRƯỚC KHI CÀI BẤT KỲ THỨ GÌ ────────────────────
# vieneu dùng ONNX external data → bị chặn bởi onnxruntime >= 1.18
print('[1/4] Đang cài đặt onnxruntime đúng version cho vieneu...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'onnxruntime', 'onnxruntime-gpu', 'onnxruntime-training'],
               capture_output=True)

# Cài 1.17.3 — version cuối TRƯỚC KHI có external data path check
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu==1.17.3'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('  ⚠️ GPU build không có, thử CPU build...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime==1.17.3'],
                   capture_output=True)

# Verify
r2 = subprocess.run([sys.executable, '-c', 'import onnxruntime; print(onnxruntime.__version__)'],
                    capture_output=True, text=True)
ort_ver = r2.stdout.strip()
print(f'  ✅ onnxruntime {ort_ver} (cần < 1.18 để vieneu hoạt động)')
if ort_ver >= '1.18':
    print('  ❌ Vẫn còn version cũ trong bộ nhớ. CẦN RESTART KERNEL:')
    print('     Thời gian chạy → Khởi động lại phiên → Chạy lại cell này')
    raise SystemExit('Cần restart kernel sau khi downgrade onnxruntime')

# ── BƯỚC 1: Cài dependencies ─────────────────────────────────────────────────
print('\n[2/4] Đang cài Flask, vieneu, cloudflared...')
for pkg in ['flask', 'flask-cors', 'soundfile', 'requests', 'pycloudflared', 'vieneu']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode == 0 else "❌"} {pkg}')

# ── BƯỚC 2: Tải worker code ──────────────────────────────────────────────────
print('\n[3/4] Đang tải worker code từ GitHub...')
import urllib.request
WORKER_URL = 'https://raw.githubusercontent.com/sleep2k0965388977-png/sleep2k/main/local_clone_worker.py'
try:
    urllib.request.urlretrieve(WORKER_URL, '/tmp/colab_worker.py')
    print(f'  ✅ Đã tải worker code')
except Exception as e:
    print(f'  ❌ Không tải được: {e}')
    raise

# ── BƯỚC 3: Tạo Cloudflare Tunnel rồi chạy worker ───────────────────────────
print('\n[4/4] Đang tạo Cloudflare Tunnel và khởi động worker...')

WORKER_PORT   = 5005
RAILWAY_URL   = os.environ.get('RAILWAY_URL', 'https://sleep2k-1.onrender.com')
CLONE_API_KEY = os.environ.get('CLONE_API_KEY', 'sleep2k_clone_2024')

# Lấy tunnel URL trước khi chạy worker
tunnel_url = None
try:
    from pycloudflared import try_cloudflare
    # Chạy worker trong subprocess để lấy tunnel URL
    import threading

    # Khởi động worker trong thread riêng
    env = os.environ.copy()
    env['CLONE_API_KEY'] = CLONE_API_KEY
    env['PORT'] = str(WORKER_PORT)
    worker_proc = subprocess.Popen(
        [sys.executable, '/tmp/colab_worker.py'],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    # Đợi worker ready
    print('  ⏳ Đang khởi động AI engine (có thể mất 1-2 phút lần đầu tải model)...')
    ready = False
    for _ in range(300):  # 5 phút timeout
        line = worker_proc.stdout.readline()
        if line:
            print(f'  [worker] {line.rstrip()}')
        if 'Worker đang lắng nghe' in line or 'Running on' in line or 'đã khởi động' in line.lower():
            ready = True
            break
        if worker_proc.poll() is not None:
            print('  ❌ Worker đã thoát sớm!')
            break
        time.sleep(1)

    if ready:
        # Tạo tunnel
        tunnel = try_cloudflare(port=WORKER_PORT, verbose=False)
        tunnel_url = tunnel.tunnel_url
        print(f'\n  ✅ Cloudflare Tunnel: {tunnel_url}')
except Exception as e:
    print(f'  ⚠️ pycloudflared lỗi: {e}')
    # Fallback: chạy cloudflared CLI
    try:
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', f'http://localhost:{WORKER_PORT}'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        for _ in range(20):
            line = cf_proc.stdout.readline()
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if m:
                tunnel_url = m.group(0)
                print(f'  ✅ Tunnel: {tunnel_url}')
                break
    except Exception as e2:
        print(f'  ❌ Cloudflared: {e2}')

# Đăng ký URL lên Railway
if tunnel_url:
    import requests
    print(f'\n📡 Đăng ký với {RAILWAY_URL}...')
    try:
        r = requests.post(
            f'{RAILWAY_URL}/api/set_clone_worker_url',
            json={'worker_url': tunnel_url},
            timeout=15
        )
        if r.status_code == 200:
            print('✅ Đã đăng ký thành công! Website sẽ dùng worker này.')
        else:
            print(f'⚠️ {r.status_code}: Tự dán URL vào ô Worker trên web')
    except Exception as e:
        print(f'⚠️ Không kết nối được: {e}')
        print(f'   👉 Tự dán URL này vào ô Worker trên website: {tunnel_url}')

print('\n' + '='*55)
print('🎉 WORKER SẴN SÀNG!')
print(f'   URL: {tunnel_url}')
print('   Giữ tab Colab mở — đừng đóng!')
print('='*55)

# Heartbeat & forward worker logs
import requests
while True:
    time.sleep(30)
    # In log từ worker
    if 'worker_proc' in dir() and worker_proc.poll() is None:
        try:
            line = worker_proc.stdout.readline()
            if line.strip():
                print(f'[worker] {line.rstrip()}')
        except:
            pass
    # Heartbeat
    try:
        requests.get(f'http://localhost:{WORKER_PORT}/health', timeout=5)
        print(f'💓 {time.strftime("%H:%M:%S")}')
    except:
        print(f'⚠️ {time.strftime("%H:%M:%S")} worker không phản hồi')